# PDF to MD Converter with Datalab Marker

## Описание инструмента

Представленный ноутбук является демонстрацией применения инструмента `marker` от `Datalab` для конвертации PDF-документов в Markdown-формат, с которым работают `LLM`. Преимущество инструмента состоит в возможности автоматической вёрстки `LaTeX` и обработки изображений на основе `OCR`.

## Использование

Установка зависимостей

In [ ]:
!python -m pip install --upgrade pip
!python -m pip install marker-pdf[full]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 73.3 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.6/948.6 kB 36.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 102.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 85.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 117.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 796.9/796.9 kB 43.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 MB 74.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 122.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 29.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Подгрузка файлов из Google Drive. Создайте папку со всеми исходными PDF-файлами. Альтернативно загрузите напрямую в папку с проектом.

In [ ]:
from google.colab import drive
import os
import shutil

drive.mount('/content/drive')

SRC_DIR = "/content/drive/MyDrive/pdfs"
DST_DIR = "/content/pdfs"
os.makedirs(DST_DIR, exist_ok=True)

for fn in os.listdir(SRC_DIR):
    if fn.lower().endswith(".pdf"):
        shutil.copy2(os.path.join(SRC_DIR, fn), os.path.join(DST_DIR, fn))


Mounted at /content/drive


Выбор устройства.

In [ ]:
import os
import torch

os.environ["TORCH_DEVICE"] = "cuda" if torch.cuda.is_available() else "cpu"

print("TORCH_DEVICE set to", os.environ["TORCH_DEVICE"])
print("torch version:", torch.__version__, "cuda available:", torch.cuda.is_available())

TORCH_DEVICE set to cuda
torch version: 2.10.0+cu128 cuda available: True


Использование `marker` от `Datalab` для конвертации `PDF2MD`. `marker` позволяет обрабатывать тысячи страниц, один его процесс `worker` расходует около 3.5 Гб VRAM. Существенным минусом инструмента является отсутствие ёprogress barё с счётчиком обработанных страниц.

In [ ]:
!marker "/content/pdfs" --output_format markdown --output_dir "/content/markdown_result" --workers 3 --disable_image_extraction

## Производительность

Инструмент тестировался на 5 томах учебника Иродова по общей физике с читаемым текстом, но утраченным `LaTeX`. Система успешно сверстала все формулы в MD.

Примерная производительность в приведённом выше варианте команды `!marker...` на `T4 GPU Colab` составила $0.15–0.21$ стр/сек для документов по $300$ страниц. В результате для каждого документа получаются два файла: текст в формате `MD` и `JSON` с данными об исходном форматировании.

Полученный результат может быть использован в `Vector DB` для `RAG`-систем и для `LLM fine-tuning`.

## Пример результата конвертации в MD

4.15. Атом массы  $m_1$  испытал неупругое столкновение с покоившейся молекулой массы  $m_2$ . После соударения обе частицы разлетелись под углом  $\vartheta$  друг к другу с кинетическими энергиями  $K_1'$  и  $K_2'$  соответственно, причем молекула оказалась в возбужденном состоянии — ее внутренняя энергия увеличилась на определенную величину Q. Найти Q, а также пороговую кинетическую энергию атома, при которой возможен переход молекулы в данное возбужденное состояние.

Р е ш е н и е. Из законов сохранения энергии и импульса в этом процессе следует:

$$\begin{split} K_1 &= K_1' + K_2' + Q, \\ p_1^2 &= {p_1'}^2 + {p_2'}^2 + 2\,p_1'\,p_2'\cos\,\vartheta, \end{split}$$

где штрихами отмечены величины после соударения (второе соотношение сразу следует из треугольника импульсов согласно теореме косинусов). Воспользовавшись формулой  $p^2=2mK$ , исключим  $K_1$  из этих уравнений. В результате получим

$$Q = (m_2/m_1 - 1)K_2' + 2\sqrt{(m_2/m_1)K_1'K_2'\cos\vartheta},$$

$$K_{1\text{nop}} = |Q|(m_1 + m_2)/m_2.$$